# Multi-Agent Orchestration with CrewAI & Google Gemini

In this notebook, we explore **Multi-Agent Systems (MAS)** using the **CrewAI** framework powered by **Google Gemini**. 

While LangChain ReAct agents excel at granular reasoning and dynamic tool-calling loops for a single brain, **CrewAI** models human organization structures. It enables multiple autonomous agents with specialized **roles**, **goals**, and **backstories** to collaborate, pass context, and execute complex workflows.

### Notebook Roadmap (Progression by Complexity):
1. **Crew 1: Simple Single-Agent Crew** — **Customer Support & Returns Specialist** — A single focused customer resolution agent equipped with custom return and warranty policy tools.
2. **Crew 2: Intermediate Multi-Agent Sequential Crew** — **Content Creation & Viral Editorial Pipeline** — A two-agent pipeline (Trend & Topic Researcher $\rightarrow$ Engaging Blog Copywriter) featuring direct **Task Context Passing** (`context=[research_task]`).
3. **Crew 3: Advanced 3-Agent Collaborative Crew** — **Vacation & Travel Trip Planner Triad** — A production triad (Travel Itinerary Architect, Destination Activity Specialist, Budget & Safety Auditor) with multi-task dependencies and automated markdown report generation (`output_file='vacation_itinerary.md'`).

### 1. Environment Setup & CrewAI LLM Configuration

To connect CrewAI to Google Gemini, we configure `GEMINI_API_KEY` (or `GOOGLE_API_KEY`). CrewAI allows native LLM configuration using its `LLM` class.

In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# gemini_key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")
# if not gemini_key:
#     raise ValueError("Please configure GEMINI_API_KEY or GOOGLE_API_KEY in your .env file.")

# os.environ["GEMINI_API_KEY"] = gemini_key
# os.environ["GOOGLE_API_KEY"] = gemini_key

In [ ]:
from crewai import LLM

# Configure LLM for CrewAI using Gemini 2.5 Flash
gemini_llm = LLM(
    model="gemini/gemini-3-flash-preview",
    # api_key=gemini_key,
    temperature=0.2
)
print("Successfully configured CrewAI Gemini LLM: gemini-2.5-flash")

---
## Crew 1: Simple Single-Agent Crew (Customer Support Specialist)

### Core Concept: The Anatomy of an Agent
In CrewAI, every agent is defined by three fundamental anchors:
- **`role`**: Defines the agent's function and expertise domain (e.g. "Customer Support & Returns Specialist").
- **`goal`**: The overarching objective guiding the agent's decisions.
- **`backstory`**: The persona narrative providing context and tone, improving prompt alignment.

Below, we build a customer service agent equipped with a custom `@tool` to look up return windows and warranty terms for purchased products.

In [ ]:
from crewai import Agent
from crewai.tools import tool

# 1. Define Custom Support Policy Tool
@tool
def lookup_return_and_warranty_policy(product_category: str) -> str:
    """
    Retrieves company return window, restocking fee, and warranty coverage for product categories.
    Categories include: 'electronics', 'appliances', 'apparel', 'books'.
    """
    category_clean = product_category.strip().lower()
    policies = {
        "electronics": {"return_window_days": 30, "restocking_fee_pct": 0, "warranty_months": 12, "condition": "Original packaging required"},
        "appliances": {"return_window_days": 45, "restocking_fee_pct": 5, "warranty_months": 24, "condition": "Proof of professional installation for major units"},
        "apparel": {"return_window_days": 60, "restocking_fee_pct": 0, "warranty_months": 3, "condition": "Unworn with tags attached"},
        "books": {"return_window_days": 14, "restocking_fee_pct": 0, "warranty_months": 0, "condition": "Undamaged, unmarked"}
    }
    
    data = policies.get(category_clean, {
        "return_window_days": 14, "restocking_fee_pct": 10, "warranty_months": 6, "condition": "Standard inspection required"
    })
    return (
        f"Policy for [{category_clean.upper()}] -> Return Window: {data['return_window_days']} days | "
        f"Restocking Fee: {data['restocking_fee_pct']}% | Warranty: {data['warranty_months']} months | "
        f"Condition: {data['condition']}"
    )

In [ ]:
# 2. Define Customer Support Agent
support_agent = Agent(
    role="Customer Support & Warranty Specialist",
    goal="Deliver helpful, empathetic, and policy-compliant resolutions to customer return and warranty inquiries.",
    backstory=(
        "You are a highly praised customer success specialist at a premier retail company. "
        "You excel at active listening, de-escalating customer frustration, and explaining policies clearly and warmly."
    ),
    tools=[lookup_return_and_warranty_policy],
    llm=gemini_llm,
    verbose=True,
    allow_delegation=False
)

print(f"Agent Created: {support_agent.role}")

In [ ]:
from crewai import Task, Crew

# Define Task for Crew 1
support_task = Task(
    description=(
        "A customer purchased a smart espresso machine under the 'appliances' category 20 days ago. "
        "The machine's pressure pump is not functioning properly. "
        "1. Use lookup_return_and_warranty_policy to check the return window and warranty for 'appliances'.\n"
        "2. Determine if the customer is eligible for a return or warranty repair.\n"
        "3. Write an empathetic, step-by-step resolution message explaining their options clearly."
    ),
    expected_output="A professional, friendly customer support resolution message with clear return/warranty instructions.",
    agent=support_agent
)

In [ ]:
# Assemble and Kickoff Crew 1
crew1 = Crew(
    agents=[support_agent],
    tasks=[support_task],
    verbose=True
)

print("=== Starting Crew 1 Kickoff (Customer Support Single-Agent) ===")
result_crew1 = crew1.kickoff()
print("\n--- Crew 1 Output Deliverable ---\n", result_crew1.raw)

---
## Crew 2: Intermediate Multi-Agent Sequential Crew (Content Creation Pipeline)

In real-world creative workflows, high-performing content requires a pipeline: first a **Researcher** identifies audience trends and core takeaways, then a **Copywriter** transforms those insights into an engaging, polished article.

### Key Concept: Sequential Context Passing (`context=[...]`)
CrewAI allows tasks to be linked sequentially (`Process.sequential`). By specifying `context=[research_task]`, Task 2 automatically receives the full output from Task 1 as input background context.

In [ ]:
from crewai import Agent
from crewai.tools import tool

# 1. Define Topic Research Tool
@tool
def fetch_trending_topic_insights(topic: str) -> str:
    """
    Retrieves audience pain points, top 3 engagement hooks, and core content pillars for a given topic.
    """
    topic_clean = topic.strip().lower()
    insights_db = {
        "remote work productivity": {
            "audience_pain": "Distractions at home, blurred work-life boundaries, video call fatigue.",
            "key_hooks": ["The 2-Minute Time Blocking Rule", "Why asynchronous communication saves 10 hrs/week", "Designing a distraction-free physical workspace"],
            "actionable_tip": "Audit your daily notifications and turn off all non-essential alerts for 4-hour focus blocks."
        },
        "healthy meal prep for beginners": {
            "audience_pain": "Lack of time during weekdays, food getting soggy, repetitive boring meals.",
            "key_hooks": ["The 2-sauce formula for infinite meal variety", "Top 5 ingredients that stay fresh for 6 days", "The 90-minute Sunday batch cooking workflow"],
            "actionable_tip": "Prep versatile bases (quinoa, roasted veggies, grilled protein) and vary flavors using different dressings."
        }
    }
    
    data = insights_db.get(topic_clean, {
        "audience_pain": "Finding reliable starting steps without feeling overwhelmed.",
        "key_hooks": ["3 Simple beginner strategies", "Common pitfalls to avoid", "Quick 15-minute daily habit"],
        "actionable_tip": "Start small with one micro-habit before adding complexity."
    })
    return (
        f"Topic Insights [{topic.upper()}]:\n"
        f"- Audience Pain Points: {data['audience_pain']}\n"
        f"- Engagement Hooks: {', '.join(data['key_hooks'])}\n"
        f"- Core Actionable Tip: {data['actionable_tip']}"
    )

# 2. Define Agent 1: Content & Trend Researcher
trend_researcher = Agent(
    role="Digital Content & Audience Trend Researcher",
    goal="Uncover reader pain points, viral engagement angles, and high-impact educational facts.",
    backstory=(
        "You are an expert audience analyst and digital trend researcher with a knack for spotting "
        "what makes topics resonate deeply with modern online readers."
    ),
    tools=[fetch_trending_topic_insights],
    llm=gemini_llm,
    verbose=True,
    allow_delegation=False
)

# 3. Define Agent 2: Engaging Copywriter
copywriter_agent = Agent(
    role="Senior Creative Copywriter & Editorial Specialist",
    goal="Craft catchy, beautifully structured, and highly engaging blog posts that delight and educate readers.",
    backstory=(
        "You are an award-winning digital writer and storyteller. You take dense research notes "
        "and transform them into conversational, punchy, and enjoyable articles with clear takeaways."
    ),
    llm=gemini_llm,
    verbose=True,
    allow_delegation=False
)

print(f"Crew 2 Agents Assembled: [{trend_researcher.role}] and [{copywriter_agent.role}]")

In [ ]:
from crewai import Process, Task, Crew

# Task 1: Topic & Audience Research
research_task = Task(
    description=(
        "Research the topic 'Remote Work Productivity'. "
        "1. Use fetch_trending_topic_insights to pull the core pain points, hooks, and actionable tips.\n"
        "2. Structure a clear outline with 3 distinct article sections and a strong opening angle."
    ),
    expected_output="A structured research brief containing audience pain points, key hooks, and a 3-part article outline.",
    agent=trend_researcher
)

# Task 2: Engaging Article Draft (receives context from Task 1)
writing_task = Task(
    description=(
        "Using the research brief provided by the trend researcher, write a complete, engaging 400-word blog article. "
        "Include an irresistible headline, introduction hook, 3 practical tips with subheadings, and a punchy conclusion."
    ),
    expected_output="A complete, beautifully formatted markdown blog article ready for publishing.",
    agent=copywriter_agent,
    context=[research_task]
)

# Assemble and Execute Sequential Crew
crew2 = Crew(
    agents=[trend_researcher, copywriter_agent],
    tasks=[research_task, writing_task],
    process=Process.sequential,
    verbose=True
)

print("=== Starting Crew 2 Kickoff (Content Creation Pipeline) ===")
result_crew2 = crew2.kickoff()
print("\n--- Crew 2 Output Deliverable ---\n", result_crew2.raw)

---
## Crew 3: Advanced 3-Agent Collaborative Crew (Vacation & Travel Trip Planner)

In this section, we build a **production triad** of 3 collaborating agents:
1. **Travel Itinerary Architect**: Outlines trip structure, daily themes, and logistics.
2. **Destination Activity Specialist**: Uses custom destination tools to curate top attractions, local dining, and ticket estimates.
3. **Budget & Safety Auditor**: Inspects total expenses against the traveler's budget, checks safety tips, and exports the final deliverable directly to disk as `vacation_itinerary.md` using `output_file`.

In [ ]:
from crewai import Agent
from crewai.tools import tool

# 1. Define Custom Destination Tool
@tool
def search_city_attractions(city: str, category: str = "highlights") -> str:
    """
    Searches top attractions, dining recommendations, and estimated costs for a destination city.
    Categories: 'highlights', 'dining', 'outdoor'.
    """
    city_clean = city.strip().lower()
    japan_db = {
        "tokyo": {
            "highlights": "1. Senso-ji Temple (Free) | 2. Shibuya Sky Observatory ($18) | 3. TeamLab Borderless ($28)",
            "dining": "Tsukiji Outer Market ($25 budget lunch), Omoide Yokocho Yakitori ($30 dinner)",
            "transit": "72-hour Tokyo Subway Pass ($10)"
        },
        "kyoto": {
            "highlights": "1. Fushimi Inari Shrine (Free) | 2. Kinkaku-ji Golden Pavilion ($4) | 3. Arashiyama Bamboo Grove (Free)",
            "dining": "Gion Kaiseki Tasting ($65), Nishiki Market Street Food ($20)",
            "transit": "Kyoto Bus 1-Day Pass ($5)"
        }
    }
    
    data = japan_db.get(city_clean, {
        "highlights": "Historic Old Town walking tour ($15), Central City Museum ($12)",
        "dining": "Local culinary bistro ($35)",
        "transit": "Day pass ($8)"
    })
    return (
        f"City Guide [{city.upper()} - {category.upper()}]:\n"
        f"- Top Attractions: {data.get('highlights', 'N/A')}\n"
        f"- Recommended Dining: {data.get('dining', 'N/A')}\n"
        f"- Local Transit Tip: {data.get('transit', 'N/A')}"
    )

# 2. Define Agent 1: Travel Itinerary Architect
trip_planner = Agent(
    role="Lead Travel Experience Designer",
    goal="Architect immersive, well-paced, and logically sequenced day-by-day travel itineraries.",
    backstory=(
        "You are a seasoned globetrotter and master travel planner who crafts unforgettable vacation schedules "
        "with optimal daily pacing so travelers never feel rushed."
    ),
    llm=gemini_llm,
    verbose=True,
    allow_delegation=False
)

# 3. Define Agent 2: Destination Activity Specialist
destination_specialist = Agent(
    role="Destination Cultural & Culinary Explorer",
    goal="Curate authentic local activities, hidden cultural gems, and delicious dining recommendations.",
    backstory=(
        "You are a local insider who knows the best spots, hidden alleyway restaurants, "
        "and ticket costs in the world's most exciting travel destinations."
    ),
    tools=[search_city_attractions],
    llm=gemini_llm,
    verbose=True,
    allow_delegation=False
)

# 4. Define Agent 3: Budget & Travel Safety Auditor
budget_safety_auditor = Agent(
    role="Senior Travel Budget & Safety Compliance Auditor",
    goal="Audit estimated total costs against traveler budget constraints and add critical travel safety advisories.",
    backstory=(
        "You are a travel finance and risk expert ensuring itineraries stay strictly on budget, "
        "with realistic daily expense allowances and essential emergency tips."
    ),
    llm=gemini_llm,
    verbose=True,
    allow_delegation=False
)

print("Crew 3 Triad Assembled:", [trip_planner.role, destination_specialist.role, budget_safety_auditor.role])

In [ ]:
from crewai import Process, Task, Crew

output_report_path = "vacation_itinerary.md"

# Task 1: Schedule Architecture
plan_task = Task(
    description=(
        "Design a 3-day vacation itinerary structure for a trip to Tokyo. "
        "The traveler has a total activity & dining budget of $300 USD. "
        "Outline Day 1: Modern City & Views, Day 2: Culture & History, Day 3: Art & Street Food."
    ),
    expected_output="A 3-day travel itinerary schedule with morning, afternoon, and evening themes.",
    agent=trip_planner
)

# Task 2: Attraction & Dining Enrichment (receives context from Task 1)
enrich_task = Task(
    description=(
        "Using search_city_attractions, look up top attractions, dining, and transit options for Tokyo. "
        "Fill in the 3-day schedule with exact attraction names, estimated ticket costs, and recommended meal spots."
    ),
    expected_output="An enriched 3-day itinerary with verified attractions, dining spots, and itemized entry fees.",
    agent=destination_specialist,
    context=[plan_task]
)

# Task 3: Budget Audit & File Export (receives context from Task 2)
audit_task = Task(
    description=(
        "Review the enriched Tokyo itinerary. "
        "1. Calculate the total estimated cost for attractions, dining, and transit.\n"
        "2. Verify if the total fits within the $300 USD budget constraint.\n"
        "3. Include emergency travel safety tips (e.g. cash usage, pocket Wi-Fi, transit cards).\n"
        "4. Output the complete final travel brief in clean markdown."
    ),
    expected_output="A comprehensive, publication-ready Tokyo 3-Day Travel Guide with itemized budget audit and safety checklist.",
    agent=budget_safety_auditor,
    context=[enrich_task],
    output_file=output_report_path
)

# Assemble Collaborative Crew Triad
crew3 = Crew(
    agents=[trip_planner, destination_specialist, budget_safety_auditor],
    tasks=[plan_task, enrich_task, audit_task],
    process=Process.sequential,
    verbose=True
)

print("=== Starting Crew 3 Kickoff (Travel Triad) ===")
result_crew3 = crew3.kickoff()
print(f"\n--- Crew 3 Completed! Saved report to: {output_report_path} ---")

In [ ]:
import os

# Inspect generated deliverable artifact
if os.path.exists(output_report_path):
    print(f"File '{output_report_path}' successfully generated!\n")
    with open(output_report_path, "r", encoding="utf-8") as f:
        content = f.read()
    print(content[:1200] + "\n... [Itinerary continues in file] ...")
else:
    print("Preview of final QA output:\n", str(result_crew3)[:1200])

### Summary & Architectural Synthesis

| Architecture | Paradigm | Strengths | Best Applied In |
| :--- | :--- | :--- | :--- |
| **LangChain ReAct** | Monolithic Autonomous Loop | Fast, dynamic tool selection, low latency, fine-grained scratchpad debugging. | **Customer Support, Real Estate Search, Clinical Calculations**, SQL querying, single-session chat assistants. |
| **CrewAI Multi-Agent** | Role-Playing Specialized Teams | Human organizational alignment, separation of concerns, task context passing, automated artifact export. | **Content Creation Pipelines, Travel & Event Planning**, editorial reviews, enterprise compliance auditing. |

> **Engineering Tip**: In production architectures, enterprise teams frequently combine both: an overarching CrewAI team where individual agents execute specialized LangChain ReAct loops to solve sub-problems!